In [0]:
# order_id STRING,
# item_id STRING,
# restaurant_id STRING,
# order_timestamp TIMESTAMP,
# order_date DATE,
# item_name STRING,
# category STRING,
# quantity INT,
# unit_price DECIMAL(10,2),
# subtotal DECIMAL(10,2),

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
items_schema = ArrayType(
    StructType(
        [
            StructField("item_id", StringType()),
            StructField("name", StringType()),
            StructField("category", StringType()),
            StructField("quantity", IntegerType()),
            StructField("unit_price", DecimalType(10, 2)),
            StructField("subtotal", DecimalType(10, 2)),
        ]
    )
)

df_fact_orders = (
    spark.read.table("`restaurantopcat`.`01_bronze`.`orders`")
    .withColumn("order_timestamp", to_timestamp(col("order_timestamp")))
    .withColumn("order_date", to_date(col("order_timestamp")))
    .withColumn("items_parsed", from_json(col("items"), items_schema))
    .withColumn("item", explode(col("items_parsed")))
    .select(
        "order_id",
        col("item.item_id").alias("item_id"),
        "restaurant_id",
        "order_timestamp",
        "order_date",
        col("item.name").alias("item_name"),
        col("item.category").alias("category"),
        col("item.quantity").alias("quantity"),
        col("item.unit_price").cast("decimal(10, 2)").alias("unit_price"),
        col("item.subtotal").cast("decimal(10, 2)").alias("subtotal"),
    )
)

display(df_fact_orders)